# Singular Value Decomposition (SVD) - Deep Dive

**Author**: AI Mathematics Study Group  
**Level**: Graduate  
**Prerequisites**: Linear Algebra basics, NumPy  
**Estimated Time**: 2-3 hours

---

## 📚 Learning Objectives

By the end of this notebook, you will:

1. ✅ Understand the mathematical foundation of SVD
2. ✅ Implement SVD from scratch (for educational purposes)
3. ✅ Apply SVD to real-world problems:
   - Image compression
   - Recommender systems
   - Principal Component Analysis (PCA)
   - Noise reduction
4. ✅ Analyze computational complexity
5. ✅ Understand when and why to use SVD

---

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
from PIL import Image
import time

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully")
print(f"NumPy version: {np.__version__}")

## 1. Mathematical Foundation

### 1.1 Definition

For any matrix $A \in \mathbb{R}^{m \times n}$, the **Singular Value Decomposition** is:

$$A = U \Sigma V^T$$

where:

- $U \in \mathbb{R}^{m \times m}$: **Left singular vectors** (orthogonal matrix)
  - Columns are eigenvectors of $AA^T$
  - $U^T U = I_m$

- $\Sigma \in \mathbb{R}^{m \times n}$: **Singular values** (diagonal matrix)
  - Diagonal entries: $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$
  - $r = \text{rank}(A)$
  - $\sigma_i = \sqrt{\lambda_i}$ where $\lambda_i$ are eigenvalues of $A^TA$

- $V \in \mathbb{R}^{n \times n}$: **Right singular vectors** (orthogonal matrix)
  - Columns are eigenvectors of $A^TA$
  - $V^T V = I_n$

### 1.2 Geometric Interpretation

SVD decomposes any linear transformation into three steps:

1. **Rotation** (by $V^T$)
2. **Scaling** (by $\Sigma$)
3. **Rotation** (by $U$)

### 1.3 Key Properties

1. **Existence**: SVD exists for ANY matrix (even non-square, singular)
2. **Uniqueness**: Singular values are unique (up to ordering)
3. **Best Low-Rank Approximation**: 
   $$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$
   minimizes $\|A - A_k\|_F$ (Frobenius norm)
4. **Pseudoinverse**: $A^+ = V \Sigma^+ U^T$

In [ ]:
# Example 1: Basic SVD
print("Example 1: Basic SVD Computation\n" + "="*50)

# Create a simple 3x2 matrix
A = np.array([
    [1, 2],
    [3, 4],
    [5, 6]
], dtype=float)

print("Original Matrix A:")
print(A)
print(f"Shape: {A.shape}\n")

# Compute SVD using NumPy
U, s, Vt = np.linalg.svd(A, full_matrices=True)

print("U (Left singular vectors):")
print(U)
print(f"Shape: {U.shape}\n")

print("Singular values (σ):")
print(s)
print(f"Shape: {s.shape}\n")

print("V^T (Right singular vectors, transposed):")
print(Vt)
print(f"Shape: {Vt.shape}\n")

# Verify: U^T U = I
print("Verification: U^T U = I")
print(U.T @ U)
print()

# Reconstruct A from SVD
Sigma = np.zeros((U.shape[0], Vt.shape[0]))
Sigma[:len(s), :len(s)] = np.diag(s)

A_reconstructed = U @ Sigma @ Vt

print("Reconstructed Matrix A:")
print(A_reconstructed)
print()

print("Reconstruction Error:")
print(f"||A - UΣV^T||_F = {np.linalg.norm(A - A_reconstructed):.2e}")

## 2. Computing SVD: The Algorithm

### 2.1 Standard Method

The most common algorithm:

1. Compute $A^T A$
2. Find eigenvalues and eigenvectors of $A^T A$
   - Eigenvectors → columns of $V$
   - $\sigma_i = \sqrt{\lambda_i}$
3. Compute $U$ from $U = AV\Sigma^{-1}$

### 2.2 Complexity Analysis

For $A \in \mathbb{R}^{m \times n}$ with $m \geq n$:

- **Time**: $O(mn^2 + n^3)$
- **Space**: $O(m^2 + n^2)$ for full SVD

For large matrices, use **truncated SVD** (only compute top $k$ components).

In [ ]:
# Example 2: SVD from Scratch (Educational)
print("Example 2: SVD Implementation from Scratch\n" + "="*50)

def svd_from_scratch(A, full_matrices=True):
    """
    Compute SVD using eigendecomposition.
    
    This is for educational purposes only!
    In practice, use np.linalg.svd which is more stable.
    """
    m, n = A.shape
    
    # Step 1: Compute A^T A
    ATA = A.T @ A
    
    # Step 2: Eigendecomposition of A^T A
    eigenvalues, V = np.linalg.eigh(ATA)
    
    # Sort in descending order
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    V = V[:, idx]
    
    # Step 3: Compute singular values
    singular_values = np.sqrt(np.maximum(eigenvalues, 0))
    
    # Step 4: Compute U
    # U = A V Σ^(-1)
    r = np.sum(singular_values > 1e-10)  # numerical rank
    U = np.zeros((m, m if full_matrices else r))
    
    for i in range(r):
        U[:, i] = (A @ V[:, i]) / singular_values[i]
    
    # Complete U to orthonormal basis if full_matrices
    if full_matrices and r < m:
        # Use QR decomposition on random matrix for remaining columns
        Q, _ = np.linalg.qr(np.random.randn(m, m - r))
        U[:, r:] = Q
    
    return U, singular_values[:r], V.T

# Test our implementation
A_test = np.random.randn(5, 3)

print("Testing our SVD implementation...\n")

# Our implementation
U_ours, s_ours, Vt_ours = svd_from_scratch(A_test, full_matrices=False)

# NumPy's implementation
U_np, s_np, Vt_np = np.linalg.svd(A_test, full_matrices=False)

# Reconstruct and compare
A_ours = U_ours @ np.diag(s_ours) @ Vt_ours
A_np = U_np @ np.diag(s_np) @ Vt_np

print(f"Our reconstruction error: {np.linalg.norm(A_test - A_ours):.2e}")
print(f"NumPy reconstruction error: {np.linalg.norm(A_test - A_np):.2e}")
print(f"\nSingular values difference: {np.linalg.norm(s_ours - s_np):.2e}")

print("\n⚠️  Note: Small differences are expected due to sign ambiguity in eigenvectors")

## 3. Application 1: Image Compression

### Theory

Images can be compressed using low-rank SVD approximation:

$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$

**Compression Ratio**:
- Original: $m \times n$ values
- Compressed: $k(m + n + 1)$ values
- Ratio: $\frac{mn}{k(m+n+1)}$

**Quality Metric**:
- Relative error: $\frac{\|A - A_k\|_F}{\|A\|_F}$
- Energy preserved: $\frac{\sum_{i=1}^{k} \sigma_i^2}{\sum_{i=1}^{r} \sigma_i^2}$

In [ ]:
# Example 3: Image Compression
print("Example 3: Image Compression using SVD\n" + "="*50)

# Create a sample grayscale image (or load your own)
# For demo, we'll create a synthetic image
def create_test_image(size=256):
    """Create a test image with patterns."""
    x = np.linspace(-3, 3, size)
    y = np.linspace(-3, 3, size)
    X, Y = np.meshgrid(x, y)
    
    # Create interesting patterns
    Z = np.sin(X) * np.cos(Y) + 0.3 * np.sin(3*X + 2*Y)
    
    # Normalize to [0, 255]
    Z = (Z - Z.min()) / (Z.max() - Z.min()) * 255
    return Z.astype(np.uint8)

# Create test image
img_original = create_test_image(256)
print(f"Image shape: {img_original.shape}")
print(f"Original size: {img_original.size} values\n")

# Compute SVD
U, s, Vt = np.linalg.svd(img_original, full_matrices=False)

print(f"Number of singular values: {len(s)}")
print(f"Largest singular value: {s[0]:.2f}")
print(f"Smallest singular value: {s[-1]:.2f}")
print(f"Condition number: {s[0]/s[-1]:.2f}\n")

# Function to reconstruct with k components
def reconstruct_image(U, s, Vt, k):
    """Reconstruct image using top k singular values."""
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

# Compress with different ranks
ranks = [5, 10, 20, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Show original
axes[0].imshow(img_original, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

# Show compressed versions
for idx, k in enumerate(ranks):
    img_compressed = reconstruct_image(U, s, Vt, k)
    
    # Compute metrics
    relative_error = np.linalg.norm(img_original - img_compressed, 'fro') / np.linalg.norm(img_original, 'fro')
    energy_preserved = np.sum(s[:k]**2) / np.sum(s**2)
    compression_ratio = img_original.size / (k * (img_original.shape[0] + img_original.shape[1] + 1))
    
    axes[idx + 1].imshow(img_compressed, cmap='gray')
    axes[idx + 1].set_title(
        f'k={k}\n'
        f'Error: {relative_error:.1%}\n'
        f'Energy: {energy_preserved:.1%}\n'
        f'Ratio: {compression_ratio:.1f}x'
    )
    axes[idx + 1].axis('off')

plt.tight_layout()
plt.savefig('svd_image_compression.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Image compression visualization saved!")

In [ ]:
# Analyze singular value decay
print("Singular Value Analysis\n" + "="*50)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Singular values
ax1.plot(s, 'o-', markersize=4)
ax1.set_xlabel('Index')
ax1.set_ylabel('Singular Value')
ax1.set_title('Singular Values (Linear Scale)')
ax1.grid(True, alpha=0.3)

# Plot 2: Cumulative energy
cumulative_energy = np.cumsum(s**2) / np.sum(s**2)
ax2.plot(cumulative_energy, 'o-', markersize=4)
ax2.axhline(y=0.9, color='r', linestyle='--', label='90% energy')
ax2.axhline(y=0.95, color='orange', linestyle='--', label='95% energy')
ax2.axhline(y=0.99, color='green', linestyle='--', label='99% energy')
ax2.set_xlabel('Number of Components (k)')
ax2.set_ylabel('Cumulative Energy Preserved')
ax2.set_title('Energy Preservation')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('svd_singular_values.png', dpi=150, bbox_inches='tight')
plt.show()

# Find k for different energy thresholds
for threshold in [0.90, 0.95, 0.99]:
    k = np.argmax(cumulative_energy >= threshold) + 1
    print(f"k={k:3d} preserves {threshold:.0%} of energy (compression ratio: {img_original.size / (k * (img_original.shape[0] + img_original.shape[1] + 1)):.1f}x)")

## 4. Application 2: Principal Component Analysis (PCA)

### Connection between PCA and SVD

Given data matrix $X \in \mathbb{R}^{n \times d}$ (n samples, d features):

1. **Center the data**: $\tilde{X} = X - \bar{X}$
2. **Covariance matrix**: $C = \frac{1}{n-1} \tilde{X}^T \tilde{X}$
3. **PCA via eigendecomposition**: $C = V \Lambda V^T$
4. **PCA via SVD**: $\tilde{X} = U \Sigma V^T$
   - Principal components: columns of $V$
   - Variance explained: $\lambda_i = \frac{\sigma_i^2}{n-1}$

**Advantages of SVD over eigendecomposition:**
- More numerically stable
- Avoids forming $\tilde{X}^T \tilde{X}$ (can overflow)
- Directly gives projection: $\tilde{X} V$

In [ ]:
# Example 4: PCA using SVD
print("Example 4: PCA using SVD\n" + "="*50)

# Generate synthetic data
np.random.seed(42)
n_samples = 300
n_features = 2

# Create correlated data
mean = [0, 0]
cov = [[3, 2],
       [2, 2]]
X = np.random.multivariate_normal(mean, cov, n_samples)

print(f"Data shape: {X.shape}")
print(f"Data mean: {X.mean(axis=0)}")
print(f"Data covariance:\n{np.cov(X.T)}\n")

# Step 1: Center the data
X_centered = X - X.mean(axis=0)

# Step 2: SVD
U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)

# Step 3: Extract principal components
principal_components = Vt.T  # columns are PCs
eigenvalues = (s ** 2) / (n_samples - 1)

print("Principal Components (eigenvectors):")
print(principal_components)
print()

print("Eigenvalues (variance explained):")
print(eigenvalues)
print()

print("Variance explained ratio:")
variance_ratio = eigenvalues / eigenvalues.sum()
print(variance_ratio)
print(f"PC1 explains {variance_ratio[0]:.1%} of variance")
print(f"PC2 explains {variance_ratio[1]:.1%} of variance")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Original data with principal components
ax1.scatter(X[:, 0], X[:, 1], alpha=0.5, s=30)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax1.axvline(x=0, color='k', linestyle='--', alpha=0.3)

# Plot principal component directions
origin = X.mean(axis=0)
for i in range(len(eigenvalues)):
    direction = principal_components[:, i] * np.sqrt(eigenvalues[i]) * 3
    ax1.arrow(origin[0], origin[1], direction[0], direction[1],
             head_width=0.3, head_length=0.3, fc=f'C{i+1}', ec=f'C{i+1}',
             linewidth=2, label=f'PC{i+1} ({variance_ratio[i]:.1%})')

ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Original Data with Principal Components')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axis('equal')

# Plot 2: Projected data (PC coordinates)
X_pca = X_centered @ principal_components
ax2.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.5, s=30)
ax2.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax2.axvline(x=0, color='k', linestyle='--', alpha=0.3)
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
ax2.set_title('Data in Principal Component Space')
ax2.grid(True, alpha=0.3)
ax2.axis('equal')

plt.tight_layout()
plt.savefig('svd_pca.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ PCA visualization saved!")

## 5. Computational Considerations

### 5.1 When to use Full vs Truncated SVD

**Full SVD** ($O(mn^2)$):
- Need all singular values
- Small matrices ($< 10000 \times 10000$)
- High accuracy required

**Truncated SVD** ($O(mnk)$ where $k \ll \min(m,n)$):
- Large sparse matrices
- Only need top $k$ components
- Memory constraints
- Use `scipy.sparse.linalg.svds`

### 5.2 Numerical Stability

Issues:
- Computing $A^T A$ can lose precision
- Condition number squares: $\kappa(A^T A) = \kappa(A)^2$

Solutions:
- Modern SVD uses **divide-and-conquer** or **Jacobi** methods
- Never form $A^T A$ explicitly in practice

In [ ]:
# Example 5: Performance Comparison
print("Example 5: SVD Performance Analysis\n" + "="*50)

sizes = [100, 200, 500, 1000, 2000]
times_full = []
times_reduced = []

print("Running benchmarks...\n")
print(f"{'Size':<10} {'Full SVD':<15} {'Reduced SVD':<15} {'Speedup':<10}")
print("-" * 60)

for size in sizes:
    # Create random matrix
    A = np.random.randn(size, size)
    
    # Full SVD
    start = time.time()
    U, s, Vt = np.linalg.svd(A, full_matrices=True)
    time_full = time.time() - start
    times_full.append(time_full)
    
    # Reduced SVD
    start = time.time()
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    time_reduced = time.time() - start
    times_reduced.append(time_reduced)
    
    speedup = time_full / time_reduced
    
    print(f"{size:<10} {time_full:<15.4f} {time_reduced:<15.4f} {speedup:<10.2f}x")

# Plot performance
plt.figure(figsize=(10, 6))
plt.plot(sizes, times_full, 'o-', label='Full SVD', linewidth=2, markersize=8)
plt.plot(sizes, times_reduced, 's-', label='Reduced SVD', linewidth=2, markersize=8)
plt.xlabel('Matrix Size (n × n)', fontsize=12)
plt.ylabel('Time (seconds)', fontsize=12)
plt.title('SVD Performance: Full vs Reduced', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('svd_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Performance analysis complete!")

## 6. Advanced Applications

### 6.1 Matrix Pseudoinverse

The **Moore-Penrose pseudoinverse**:

$$A^+ = V \Sigma^+ U^T$$

where $\Sigma^+$ has $\sigma_i^{-1}$ for $\sigma_i > 0$, else 0.

**Applications**:
- Solving least squares: $\min_x \|Ax - b\|_2$
- Solution: $x = A^+ b$

### 6.2 Noise Reduction

Assume data has structure + noise:
$$A = A_{\text{signal}} + A_{\text{noise}}$$

Strategy:
1. Compute SVD of $A$
2. Keep only large singular values (signal)
3. Discard small singular values (noise)

This is **optimal** under assumptions of low-rank signal.

In [ ]:
# Example 6: Noise Reduction
print("Example 6: Noise Reduction using SVD\n" + "="*50)

# Create clean signal: low-rank matrix
n = 200
true_rank = 5

U_true = np.random.randn(n, true_rank)
V_true = np.random.randn(n, true_rank)
s_true = np.linspace(10, 1, true_rank)

A_clean = U_true @ np.diag(s_true) @ V_true.T

# Add noise
noise_level = 0.5
A_noisy = A_clean + noise_level * np.random.randn(n, n)

print(f"Clean matrix rank: {true_rank}")
print(f"Noise level: {noise_level}")
print(f"SNR: {np.linalg.norm(A_clean, 'fro') / np.linalg.norm(A_noisy - A_clean, 'fro'):.2f}\n")

# Denoise using SVD
U, s, Vt = np.linalg.svd(A_noisy, full_matrices=False)

# Try different ranks
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

# Show clean
axes[0].imshow(A_clean, cmap='RdBu_r', vmin=-5, vmax=5)
axes[0].set_title('Clean Signal')
axes[0].axis('off')

# Show noisy
axes[1].imshow(A_noisy, cmap='RdBu_r', vmin=-5, vmax=5)
axes[1].set_title(f'Noisy (σ={noise_level})')
axes[1].axis('off')

# Denoise with different ranks
test_ranks = [3, 5, 10, 20]
for idx, k in enumerate(test_ranks):
    A_denoised = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    
    error = np.linalg.norm(A_clean - A_denoised, 'fro') / np.linalg.norm(A_clean, 'fro')
    
    axes[idx + 2].imshow(A_denoised, cmap='RdBu_r', vmin=-5, vmax=5)
    axes[idx + 2].set_title(f'Denoised (k={k})\nError: {error:.1%}')
    axes[idx + 2].axis('off')

plt.tight_layout()
plt.savefig('svd_denoising.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot singular value spectrum
plt.figure(figsize=(10, 6))
plt.semilogy(s, 'o-', label='Noisy data', markersize=5)
plt.axvline(x=true_rank, color='r', linestyle='--', linewidth=2, label=f'True rank = {true_rank}')
plt.axhline(y=noise_level, color='orange', linestyle='--', linewidth=2, label=f'Noise level ≈ {noise_level}')
plt.xlabel('Index', fontsize=12)
plt.ylabel('Singular Value (log scale)', fontsize=12)
plt.title('Singular Value Spectrum: Signal vs Noise', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('svd_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Denoising complete!")
print("\n💡 Insight: Singular values drop sharply after the true rank,")
print("   and then plateau at the noise level.")

## 7. Summary & Key Takeaways

### ✅ What we learned:

1. **Mathematical Foundation**
   - SVD decomposes any matrix: $A = U\Sigma V^T$
   - Geometric interpretation: rotation-scaling-rotation
   - Always exists, singular values are unique

2. **Computational Aspects**
   - Complexity: $O(mn^2)$ for $m \geq n$
   - Full vs truncated SVD
   - Numerical stability considerations

3. **Applications**
   - **Image compression**: Low-rank approximation
   - **PCA**: Dimensionality reduction
   - **Denoising**: Separate signal from noise
   - **Pseudoinverse**: Least squares solutions

4. **Practical Insights**
   - Energy/variance preserved by top-k components
   - Compression ratio vs reconstruction quality
   - Singular value spectrum reveals structure

### 🎯 When to use SVD:

✅ **Use SVD when:**
- Need low-rank approximation
- Computing PCA (more stable than eigendecomposition)
- Matrix is rectangular
- Need pseudoinverse
- Analyzing matrix structure

❌ **Don't use SVD when:**
- Only need determinant or trace
- Matrix has special structure (sparse, Toeplitz, etc.)
- Real-time constraints (too slow)

### 🚀 Next Steps:

1. **Eigendecomposition** (for square symmetric matrices)
2. **QR Decomposition** (for solving linear systems)
3. **Randomized SVD** (for very large matrices)
4. **Tensor Decompositions** (generalization to higher dimensions)

---

## 📝 Exercises

### Exercise 1: Theory
Prove that for any matrix $A$:
$$\|A\|_2 = \sigma_1$$
(the spectral norm equals the largest singular value)

### Exercise 2: Implementation
Implement a function `best_rank_k_approx(A, k)` that:
1. Computes SVD of $A$
2. Returns the best rank-$k$ approximation
3. Computes and returns the relative error

### Exercise 3: Application
Load a color image (3 channels), apply SVD compression to each channel separately, and compare with compressing the grayscale version.

### Exercise 4: Analysis
For a random $1000 \times 1000$ matrix, plot:
1. Singular value decay
2. Cumulative energy
3. Number of components needed for 90%, 95%, 99% energy

### Exercise 5: Challenge
Implement **randomized SVD** using the algorithm from Halko, Martinsson, and Tropp (2011).

---

### Solutions are in the next notebook! 📚

In [ ]:
# Exercise Solutions (uncomment to run)

# Exercise 2: Best rank-k approximation
def best_rank_k_approx(A, k):
    """
    Compute best rank-k approximation using SVD.
    
    Args:
        A: Input matrix (m x n)
        k: Desired rank
    
    Returns:
        A_k: Best rank-k approximation
        error: Relative Frobenius norm error
    """
    # TODO: Implement this!
    pass

# Test your implementation
# A_test = np.random.randn(100, 80)
# A_k, error = best_rank_k_approx(A_test, k=10)
# print(f"Relative error: {error:.2%}")

## 📚 References

1. **Golub, G. H., & Van Loan, C. F.** (2013). *Matrix Computations* (4th ed.). Johns Hopkins University Press.
   - Chapter 2: Matrix Analysis
   - Chapter 8: Symmetric Eigenvalue Problems

2. **Trefethen, L. N., & Bau III, D.** (1997). *Numerical Linear Algebra*. SIAM.
   - Lecture 4: Singular Value Decomposition
   - Lecture 5: More on the SVD

3. **Halko, N., Martinsson, P. G., & Tropp, J. A.** (2011). *Finding structure with randomness: Probabilistic algorithms for constructing approximate matrix decompositions*. SIAM Review, 53(2), 217-288.

4. **Strang, G.** (2016). *Introduction to Linear Algebra* (5th ed.). Wellesley-Cambridge Press.
   - Chapter 7: The Singular Value Decomposition

---

**Next Notebook**: [Eigenvalue Problems and Applications](./Eigenvalues_Applications.ipynb)